# Load data

In [ ]:
import pandas as pd

df_train = pd.read_csv('../data/processed/train_movie.csv')
df_test = pd.read_csv('../data/processed/test_movie.csv')
df_train.head()

In [ ]:
df_train.shape

(3499, 6)

In [ ]:
df_train.isnull().sum()

,0
budget,300
revenue,0
release_date,6
genres,98
runtime,36
director,47


In [ ]:
df_test.isnull().sum()

,0
budget,83
revenue,0
release_date,0
genres,32
runtime,10
director,14


# Xử lý missing value

## Release_date

In [ ]:
# Xóa giá trị null
df_train = df_train.dropna(subset=["release_date"])
df_test  = df_test.dropna(subset=["release_date"])

In [ ]:
df_train.isnull().sum()

,0
budget,300
revenue,0
release_date,0
genres,98
runtime,36
director,47


## Genres

In [ ]:
# Điền 'Unknown' cho các dòng null
df_train["genres"] = df_train["genres"].fillna("Unknown")
df_test["genres"]  = df_test["genres"].fillna("Unknown")

## Runtime

In [ ]:
# Tính median runtime theo director chỉ từ train
director_runtime_median = (
    df_train.groupby("director")["runtime"]
    .median()
)

# Tính global median từ train để fallback
global_runtime_median = df_train["runtime"].median()

def impute_runtime(df, director_median, global_median):
    missing = df["runtime"].isna()
    df.loc[missing, "runtime"] = (
        df.loc[missing, "director"]
        .map(director_median)
        .fillna(global_median)  # director không có trong train → dùng global
    )
    return df

df_train = impute_runtime(df_train, director_runtime_median, global_runtime_median)
df_test  = impute_runtime(df_test,  director_runtime_median, global_runtime_median)

## Budget

In [ ]:
# Tạo flag trước khi impute
df_train["has_budget"] = df_train["budget"].isna().astype(int)
df_test["has_budget"]  = df_test["budget"].isna().astype(int)

In [ ]:
# Chuẩn bị features cho imputer
# release_year extract từ release_date (đã clean)
df_train["release_date"] = pd.to_datetime(
    df_train["release_date"],
    errors="coerce"
)

df_test["release_date"] = pd.to_datetime(
    df_test["release_date"],
    errors="coerce"
)
df_train["release_year"] = df_train["release_date"].dt.year
df_test["release_year"]  = df_test["release_date"].dt.year

# Frequency encoding director — tính trên train
director_freq = df_train["director"].value_counts()
df_train["director_freq"] = df_train["director"].map(director_freq).fillna(1)
df_test["director_freq"]  = df_test["director"].map(director_freq).fillna(1)

# Genres MLB — fit trên train
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_lists_train = df_train["genres"].apply(lambda x: x.split("|"))
genre_lists_test  = df_test["genres"].apply(lambda x: x.split("|"))

genre_train = pd.DataFrame(
    mlb.fit_transform(genre_lists_train),
    columns=mlb.classes_, index=df_train.index
)
genre_test = pd.DataFrame(
    mlb.transform(genre_lists_test),
    columns=mlb.classes_, index=df_test.index
)

# Gộp features để impute
impute_cols = ["budget", "runtime", "release_year", "director_freq"]

impute_train = pd.concat([df_train[impute_cols], genre_train], axis=1)
impute_test  = pd.concat([df_test[impute_cols],  genre_test],  axis=1)

# Imputer — fit chỉ trên train
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10,
    random_state=42
)

imputed_train = imputer.fit_transform(impute_train)
imputed_test  = imputer.transform(impute_test)

# Chỉ fill rows bị missing
train_missing = df_train["budget"].isna()
test_missing  = df_test["budget"].isna()

df_train.loc[train_missing, "budget"] = imputed_train[train_missing, 0]
df_test.loc[test_missing,   "budget"] = imputed_test[test_missing,   0]

# Clip giá trị âm
df_train["budget"] = df_train["budget"].clip(lower=0)
df_test["budget"]  = df_test["budget"].clip(lower=0)

In [ ]:
df_train = df_train.drop(columns=["release_year", "director_freq"])
df_test  = df_test.drop(columns=["release_year", "director_freq"])

## Director

In [ ]:
# Đã xử lý điền thủ công ở file 3.

# Tạo flag
df_train["director_missing"] = df_train["director"].isna().astype(int)
df_test["director_missing"]  = df_test["director"].isna().astype(int)

# Điền unknown cho những giá trị còn NaN
df_train["director"] = df_train["director"].fillna("Unknown")
df_test["director"]  = df_test["director"].fillna("Unknown")

# Sau xử lý missing value

In [ ]:
df_train.head()

,budget,revenue,release_date,genres,runtime,director,has_budget,director_missing
0,4.535164e+07,19.0,2024-09-18,Unknown,2.0,Caleb Jonathan Sherwood,1,0
1,3.203900e+05,320390.0,2025-01-17,Crime|Drama|Thriller,96.0,J.C. Lee,0,0
2,2.866783e+06,3481520.0,2025-08-29,Action|Comedy|Horror,102.0,Macon Blair,0,0
3,1.167563e+07,11675627.0,2025-11-07,Biography|Drama|History,103.0,Cyrus Nowrasteh,0,0
4,5.300000e+01,13.0,2022-05-13,War|Action,5.0,Dan Levy,0,0


In [ ]:
print(df_train.shape)
print(df_test.shape)

(3493, 8)
(875, 8)


In [ ]:
df_train.isnull().sum()

,0
budget,0
revenue,0
release_date,0
genres,0
runtime,0
director,0
has_budget,0
director_missing,0


In [ ]:
df_test.isnull().sum()

,0
budget,0
revenue,0
release_date,0
genres,0
runtime,0
director,0
has_budget,0
director_missing,0


# Lưu file

In [ ]:
df_train.to_csv("../data/processed/train_missing_handled.csv", index=False)
df_test.to_csv("../data/processed/test_missing_handled.csv",   index=False)